# 第五章：PyTorch 模型定义

这一章关注模型怎么写。核心是 `nn.Module`，常见组织方式有 `Sequential`、`ModuleList`、`ModuleDict`。这个 notebook 会用小输入把每种写法都跑一遍。

## 1. 导入库并准备输入

我们用一个形状为 `[batch_size, 784]` 的假输入，模拟 28x28 图片展平后的向量。

In [ ]:
import copy
from collections import OrderedDict
from pathlib import Path
import tempfile

import torch
import torch.nn as nn

torch.manual_seed(42)
x = torch.randn(4, 784)
print("输入形状：", x.shape)


## 2. Sequential：最直接的顺序模型

如果模型就是“一层接一层”，`nn.Sequential` 最省事。

In [ ]:
net1 = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
)

out1 = net1(x)
print(net1)
print("输出形状：", out1.shape)


## 3. Sequential + OrderedDict：给每层起名字

名字清晰后，打印模型和定位层会更方便。

In [ ]:
net2 = nn.Sequential(OrderedDict([
    ("fc1", nn.Linear(784, 256)),
    ("relu1", nn.ReLU()),
    ("fc2", nn.Linear(256, 10)),
]))

out2 = net2(x)
print(net2)
print("输出形状：", out2.shape)
print("按名字访问第一层：", net2.fc1)


## 4. ModuleList：保存一组层，但不自动定义 forward

`ModuleList` 像 Python list，但它知道里面放的是模型层，会帮你注册参数。注意：它本身不是完整网络，不能直接调用。

In [ ]:
layers = nn.ModuleList([nn.Linear(784, 256), nn.ReLU()])
layers.append(nn.Linear(256, 10))

print(layers)
try:
    layers(x)
except NotImplementedError as exc:
    print("ModuleList 不能直接调用，需要自己写 forward：", type(exc).__name__)


## 5. 用 ModuleList 写一个真正的模型

关键是继承 `nn.Module`，然后在 `forward` 里明确每一层怎么执行。

In [ ]:
class ModuleListNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 10),
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

net3 = ModuleListNet()
out3 = net3(x)
print(net3)
print("输出形状：", out3.shape)


## 6. ModuleDict：按名字保存一组层

当你需要根据条件选择不同层时，`ModuleDict` 很方便。

In [ ]:
class ModuleDictNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleDict({
            "small": nn.Linear(784, 64),
            "large": nn.Linear(784, 256),
        })
        self.heads = nn.ModuleDict({
            "small": nn.Linear(64, 10),
            "large": nn.Linear(256, 10),
        })
        self.act = nn.ReLU()

    def forward(self, x, mode="small"):
        x = self.act(self.layers[mode](x))
        return self.heads[mode](x)

net4 = ModuleDictNet()
print("small 输出：", net4(x, mode="small").shape)
print("large 输出：", net4(x, mode="large").shape)


## 7. 修改模型中的某一层

迁移学习里经常会把预训练模型的最后分类层替换成自己的类别数。这里用一个小模型演示。

In [ ]:
old_model = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 1000),
)

new_model = copy.deepcopy(old_model)
new_model[-1] = nn.Linear(128, 3)

print("旧模型最后一层：", old_model[-1])
print("新模型最后一层：", new_model[-1])
print("新模型输出形状：", new_model(x).shape)


## 8. 冻结部分参数

冻结参数就是把 `requires_grad` 设为 `False`，训练时不会更新这些参数。

In [ ]:
for name, param in new_model.named_parameters():
    if name.startswith("0"):
        param.requires_grad = False

for name, param in new_model.named_parameters():
    print(name, "requires_grad=", param.requires_grad)


## 9. 保存和读取模型权重

实际项目更推荐保存 `state_dict`，而不是直接保存整个模型对象。

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "tiny_model_state_dict.pth"
    torch.save(new_model.state_dict(), path)

    loaded_model = nn.Sequential(
        nn.Linear(784, 128),
        nn.ReLU(),
        nn.Linear(128, 3),
    )
    loaded_model.load_state_dict(torch.load(path, map_location="cpu"))

    diff = (new_model(x) - loaded_model(x)).abs().max().item()
    print("保存再读取后的最大输出差异：", diff)


## 10. 小结

- `Sequential`：适合简单顺序结构。
- `ModuleList`：适合重复层或动态层，但要自己写 `forward`。
- `ModuleDict`：适合按名字/条件选择子模块。
- `state_dict`：保存模型权重的推荐方式。